In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.exceptions import ConvergenceWarning
import time
import psutil
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
# Constants
RANDOM_STATE = 1082

SYNTH_GENERATORS = ["ctgan", "gc", "tvae"]
SAMPLE_SIZES = [10_000, 100_000, 1_000_000]
SYNTH_FILES = [f"spoofing_{g}_{n}.csv" for g in SYNTH_GENERATORS for n in SAMPLE_SIZES]
SYNTH_FILES += ["syntheticSpoofing_cgan_data.csv", "syntheticSpoofing_gan_data.csv"]


In [5]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(objective='binary:logistic', random_state=RANDOM_STATE, eval_metric='mae', )
}

In [ ]:
import gc

results = []
for synth_name in SYNTH_FILES:
    try:
        EXP = protocol.synthetic_vs_full_real("uav", synth_name)
    except FileNotFoundError:
        print(f"Skipping {synth_name}: file not found")
        continue
    print(f"Training on synthetic: {synth_name}, testing on real data")
    for model_name, model in models.items():
        print(f"\n{model_name}")
        start_time = time.time()
        model.fit(EXP.X_train, EXP.y_train)
        train_time = time.time() - start_time

        y_pred = model.predict(EXP.X_test)
        acc = accuracy_score(EXP.y_test, y_pred)
        f1 = f1_score(EXP.y_test, y_pred, average='weighted', zero_division=0)
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(classification_report(EXP.y_test, y_pred, zero_division=0))

        num_epochs = model.n_iter_ if hasattr(model, "n_iter_") else "N/A"
        _row = evaluate(
            EXP.y_test,
            y_pred,
            model_name=model_name,
            family="Synthetic",
            y_proba=model.predict_proba(EXP.X_test) if hasattr(model, "predict_proba") else None,
            train_time_sec=train_time,
            num_epochs=num_epochs,
            parameters=str(model.get_params()),
            timing=time_inference(model.predict, EXP.X_test),
            **EXP.eval_kwargs(),
        )
        print(summary(_row))
        results.append(_row)
    del EXP
    gc.collect()


In [ ]:
from gps_ids.schema import results_path

append_validated(results, str(results_path("uav")))
print(f"Results appended to {results_path('uav')}")
